# MARE Developer Playground

This notebook is a lightweight way to explore MARE from a local clone.

It stays on the core install path by default:

- built-in PDF ingestion
- built-in lexical and object-aware retrieval
- page/snippet/highlight evidence inspection

Recommended setup from the repo root:

```bash
pip install -e ".[dev]"
pip install notebook
jupyter notebook examples/developer_playground.ipynb
```

This notebook assumes you are working from a clone of the repository.
It is not part of the installed PyPI package.
The repository includes a few tiny bundled sample PDFs in `examples/sample_pdfs/` for quick testing.


In [ ]:
from pathlib import Path

from mare import MAREApp


## Pick a PDF

By default, this notebook points at one of the bundled example PDFs in `examples/sample_pdfs/`.
You can swap `pdf_path` to your own local PDF file at any time.


In [ ]:
repo_root = Path.cwd()
sample_dir = repo_root / "examples" / "sample_pdfs"

candidate_pdfs = [
    sample_dir / "device_setup_guide.pdf",
    sample_dir / "retrieval_benchmark_note.pdf",
    sample_dir / "support_troubleshooting_card.pdf",
]

pdf_path = next((path for path in candidate_pdfs if path.exists()), None)
if pdf_path is None:
    raise FileNotFoundError("No bundled sample PDF found. Point pdf_path at a local PDF file and rerun.")

# To use your own file instead, uncomment and edit the next line.
# pdf_path = Path("/absolute/path/to/your/document.pdf")

pdf_path


## Build or reuse a local corpus

This will create `generated/<pdf-stem>.json` plus rendered page images if needed.


In [ ]:
app = MAREApp.from_pdf(pdf_path, reuse=True)
app.corpus_path


## Ask a question

Start with concrete, evidence-friendly questions, especially for manuals and support PDFs.


In [ ]:
query = "how do I connect the AC adapter"
explanation = app.explain(query, top_k=3)
best = explanation.fused_results[0] if explanation.fused_results else None

{
    "query": query,
    "intent": explanation.plan.intent,
    "selected_modalities": [item.value for item in explanation.plan.selected_modalities],
    "confidence": explanation.plan.confidence,
    "result_count": len(explanation.fused_results),
}


## Inspect the best evidence hit


In [ ]:
if best is None:
    print("No matching page found.")
else:
    print("Page:", best.page)
    print("Score:", best.score)
    print("Object type:", best.object_type or "page")
    print("Reason:", best.reason)
    print("Snippet:\n", best.snippet)
    print("Page image:", best.page_image_path)
    print("Highlight image:", best.highlight_image_path)


## Show the highlighted evidence image when available


In [ ]:
from IPython.display import Image, display

if best is not None:
    image_path = Path(best.highlight_image_path or best.page_image_path)
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print("No image file available for display.")


## Inspect extracted objects on the winning page

This is useful when debugging why a page or snippet won.


In [ ]:
if best is not None:
    objects = app.get_page_objects(best.doc_id, limit=10)
    [
        {
            "object_id": obj.object_id,
            "object_type": obj.object_type.value,
            "metadata": obj.metadata,
            "content": obj.content[:240],
        }
        for obj in objects
    ]


## Try a few manual-style questions


In [ ]:
queries = [
    "how do I connect the AC adapter",
    "how do I configure wake on lan",
    "show me the comparison table",
    "partially reinstall the set screws if they fall out",
]

results = []
for q in queries:
    hit = app.best_match(q, top_k=3)
    results.append(
        {
            "query": q,
            "page": hit.page if hit else None,
            "object_type": hit.object_type if hit else None,
            "score": hit.score if hit else None,
            "snippet": (hit.snippet[:160] + "...") if hit and hit.snippet and len(hit.snippet) > 160 else (hit.snippet if hit else None),
        }
    )

results


## Where to go next

Once the core path makes sense, inspect these files:

- `src/mare/api.py`
- `src/mare/engine.py`
- `src/mare/retrievers/text.py`
- `src/mare/objects.py`
- `src/mare/highlight.py`
- `src/mare/extensions.py`

If you want advanced retrieval or integrations, install the matching extras later rather than all at once.
